# NB11 UTXO와 계정 — 원장이 「잔액」을 적는 두 방식

『법률가를 위한 블록체인 — 기술과 경제학으로 읽기』의 장별 노트북입니다. 본문의 코드와 같습니다.

- 모든 코드는 원리를 눈으로 보기 위한 **장난감 구현**입니다. 실제 자산·실제 개인키·실제 거래에 쓰지 마십시오(실사용 금지).
- 외부 API를 부르지 않습니다. 네트워크 없이 실행됩니다. 동봉 스냅샷 데이터가 필요한 셀은 그 데이터를 셀 안에 적어 두었습니다.
- 표준 라이브러리만 씁니다(Python 3.13에서 실행·확인). 설치 셀은 없습니다.
- 출력은 엮은이가 실행한 결과를 그대로 담아 두었습니다. 다시 실행하면 같은 값이 나와야 합니다(난수는 시드를 고정했습니다).

In [ ]:
# NB11-1: UTXO 상태 전이 — 입력이 가리키는 출력을 집합에서 빼고 새 출력을 더한다 (공부용)
import hashlib, json

def txid(tx: dict) -> str:
    return hashlib.sha256(json.dumps(tx, sort_keys=True).encode()).hexdigest()[:12]

def apply_utxo(S: dict, tx: dict):
    """S: {(txid, index): {"owner":..., "value":...}}.  tx: {"inputs":[(txid, index, signer)], "outputs":[(owner, value)]}
    부테린(2014)의 APPLY 세 단계 그대로. 성공하면 (새 상태, 수수료), 실패하면 오류 문자열."""
    S = dict(S)
    in_sum = 0
    for (prev_id, idx, signer) in tx["inputs"]:
        key = (prev_id, idx)
        if key not in S:
            return f"오류: 입력 {key} 는 미사용 출력이 아니다(이미 쓰였거나 없음)"
        if S[key]["owner"] != signer:
            return f"오류: 입력 {key} 의 소유자는 {S[key]['owner']} 인데 서명자가 {signer}"
        in_sum += S[key]["value"]
    out_sum = sum(v for _, v in tx["outputs"])
    if in_sum < out_sum:
        return f"오류: 입력 합 {in_sum} < 출력 합 {out_sum}"
    for (prev_id, idx, _) in tx["inputs"]:
        del S[(prev_id, idx)]
    tid = txid(tx)
    for i, (owner, value) in enumerate(tx["outputs"]):
        S[(tid, i)] = {"owner": owner, "value": value}
    return S, in_sum - out_sum

def balance(S, who):                       # 「잔액」은 원장에 없다 — 그때그때 더한다
    return sum(o["value"] for o in S.values() if o["owner"] == who)

coinbase = {"inputs": [], "outputs": [("Alice", 10)]}          # 블록 보상으로 Alice 에게 10
S = {(txid(coinbase), 0): {"owner": "Alice", "value": 10}}
print("처음 UTXO 집합:", {k: v for k, v in S.items()})

tx1 = {"inputs": [(txid(coinbase), 0, "Alice")], "outputs": [("Bob", 3), ("Alice", 6.9)]}   # 3 지급 + 6.9 거스름돈, 0.1 수수료
S, fee = apply_utxo(S, tx1)
print("tx1 적용 후 — 잔액 Alice:", balance(S, "Alice"), "Bob:", balance(S, "Bob"), "| 수수료:", round(fee, 2))
print("UTXO 집합 크기:", len(S), "(10짜리 출력 하나가 사라지고 3, 6.9 두 개가 생겼다)")

tx2 = {"inputs": [(txid(coinbase), 0, "Alice")], "outputs": [("Carol", 10)]}                 # 같은 출력을 다시 쓰려는 이중지불
print("tx2(이중지불):", apply_utxo(S, tx2))
tx3 = {"inputs": [(txid(tx1), 0, "Mallory")], "outputs": [("Mallory", 3)]}                 # 남의 출력
print("tx3(남의 출력):", apply_utxo(S, tx3))
tx4 = {"inputs": [(txid(tx1), 0, "Bob")], "outputs": [("Carol", 3)]}                       # Bob 이 받은 3 을 Carol 에게
S, fee = apply_utxo(S, tx4)
print("tx4 적용 후 — Bob:", balance(S, "Bob"), "Carol:", balance(S, "Carol"))

처음 UTXO 집합: {('02c83c8a0105', 0): {'owner': 'Alice', 'value': 10}}
tx1 적용 후 — 잔액 Alice: 6.9 Bob: 3 | 수수료: 0.1
UTXO 집합 크기: 2 (10짜리 출력 하나가 사라지고 3, 6.9 두 개가 생겼다)
tx2(이중지불): 오류: 입력 ('02c83c8a0105', 0) 는 미사용 출력이 아니다(이미 쓰였거나 없음)
tx3(남의 출력): 오류: 입력 ('99736848c857', 0) 의 소유자는 Bob 인데 서명자가 Mallory
tx4 적용 후 — Bob: 0 Carol: 3


In [ ]:
# NB11-2: 계정 상태 전이 — 잔고를 빼고 더하되, 논스로 「같은 거래 두 번」을 막는다 (공부용)
def apply_account(S: dict, tx: dict):
    """S: {address: {"nonce": n, "balance": b}}.  tx: {"from", "to", "value", "nonce"}"""
    S = {k: dict(v) for k, v in S.items()}
    sender = S.setdefault(tx["from"], {"nonce": 0, "balance": 0})
    if tx["nonce"] != sender["nonce"]:
        return f"오류: {tx['from']} 의 다음 논스는 {sender['nonce']} 인데 거래 논스가 {tx['nonce']}"
    if sender["balance"] < tx["value"]:
        return f"오류: 잔고 {sender['balance']} < 송금액 {tx['value']}"
    sender["nonce"] += 1
    sender["balance"] -= tx["value"]
    S.setdefault(tx["to"], {"nonce": 0, "balance": 0})["balance"] += tx["value"]
    return S

S = {"Alice": {"nonce": 0, "balance": 10}}
t1 = {"from": "Alice", "to": "Bob", "value": 3, "nonce": 0}
S = apply_account(S, t1)
print("t1 적용 후:", S)
print("같은 t1 을 다시 적용(재전송 공격):", apply_account(S, t1))
t2 = {"from": "Alice", "to": "Carol", "value": 3, "nonce": 1}
S = apply_account(S, t2)
print("t2 적용 후:", S, "— 거스름돈 출력이 없다. 잔고가 그냥 줄어든다")
t3 = {"from": "Alice", "to": "0x000…dead", "value": 4, "nonce": 2}      # 존재하지 않는(열쇠 없는) 주소로 — 착오 이체
S = apply_account(S, t3)
print("t3(착오 이체) 적용 후:", S, "— 받는 계정이 자동으로 생기고 잔고가 적힌다. 되돌릴 거래는 그 주소의 열쇠만 만들 수 있다")

t1 적용 후: {'Alice': {'nonce': 1, 'balance': 7}, 'Bob': {'nonce': 0, 'balance': 3}}
같은 t1 을 다시 적용(재전송 공격): 오류: Alice 의 다음 논스는 1 인데 거래 논스가 0
t2 적용 후: {'Alice': {'nonce': 2, 'balance': 4}, 'Bob': {'nonce': 0, 'balance': 3}, 'Carol': {'nonce': 0, 'balance': 3}} — 거스름돈 출력이 없다. 잔고가 그냥 줄어든다
t3(착오 이체) 적용 후: {'Alice': {'nonce': 3, 'balance': 0}, 'Bob': {'nonce': 0, 'balance': 3}, 'Carol': {'nonce': 0, 'balance': 3}, '0x000…dead': {'nonce': 0, 'balance': 4}} — 받는 계정이 자동으로 생기고 잔고가 적힌다. 되돌릴 거래는 그 주소의 열쇠만 만들 수 있다
